### ENSEMBLE METHODS

Combining models from different families through stacking and voting
1. Voting - combine predictions by averaging (soft) or majority vote (hard) - using trained models saved in 
2. Stacking - train a meta-model that learns how to combine the base models’ predictions.

Compute out-of-fold predictions for all candidate models.
Check the correlation matrix of those predictions. Drop models that are highly correlated with a better one.
Keep the top few per family, or use greedy forward selection (add the model that most improves ensemble CV score; repeat until no gain).
Compare: best single model vs. voting vs. stacking, using the same CV. Use a held-out test set for the final comparison.

||Voting|Stacking|
| :--- | :---: | ---: |
|Complexity |Low|Higher|
|Overfitting risk|Low|Moderate (more models → more risk)|
|Gain|Reliable, modest|	Often larger if models are diverse|
|Best when|	Models are similarly strong|	Models have different strengths

Importing all predictions on validation

everyone validate on sarcasm random + sarcasm subreddit, sarcasm temporal and news headlines 
voting is on validation 

shouldnt be on test data - unseen
use validation data 

need probability score and actual predictions for each row 

4 models results r quite similar -> can use voting
subreddit - make it worse -> explanation: affect it's ability to ?? 

lgbm - for prediction
ebm for explanation
shap for feature explanation

0.001 - subreddit very low contribution to prediction
0.001 - parent thread contribution 

cannot test data 
author say overlap between train and test data, same context have diff meaning

lgbm xgboost log_reg , linear svm - voting

all voting and stacking done with the manual way - not using votingclassifier model becuase its from prediction results not from models






In [83]:
#import libraries

import pandas as pd
import numpy as  np
from scipy.stats import mode
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)

In [84]:
#import validation dataset 
#sarc_random_validation
#define routes


#validation and test data
val  = pd.read_parquet("../../data/splits/primary/sarc_random/val.parquet")
test  = pd.read_parquet("../../data/splits/primary/sarc_random/test.parquet")
y_val = val['label']
y_test = test['label']

#sarcasm_random validation predictions (probabilties)
#lite gbm /xgboost
val_lgbm = "../../results/gbdt/ensemble_predictions/sarc_random__val.parquet"
#margin-based (linear svm)
val_margin = "../../results/margin-based/ensemble_predictions/sarc_random__val.parquet"
#log_reg
val_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_random__val.parquet"

#sarcasm_random test predictions (probabilties)
#lite gbm /xgboost
test_lgbm = "../../results/gbdt/ensemble_predictions/sarc_random__test.parquet"
#margin-based (linear svm)
test_margin = "../../results/margin-based/ensemble_predictions/sarc_random__test.parquet"
#log_reg
test_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_random__test.parquet"

#model names in columns
lgbm_model = 'sarc_random__lgbm_reply_tfidf_surface'
xgb_model = 'sarc_random__xgb_reply_tfidf_surface'
lr_model = 'sarc_random__lr_reply_tfidf_surface'
svm_model = 'sarc_random__svm_tfidf_lex_surf'

display(pd.read_parquet(val_lgbm))





,row_id,row_position,label,sarc_random__lgbm_reply_tfidf,sarc_random__lgbm_reply_tfidf_surface,sarc_random__xgb_reply_tfidf,sarc_random__xgb_reply_tfidf_surface
0,8c5622f4b8edbd4fc1bb29d1a2739d90,0,0,0.349346,0.379181,0.549932,0.441270
1,ef194a1d7209570a5779540806d0be7f,1,0,0.520090,0.425580,0.462667,0.385260
2,0b6b7d6b88a0165663241c9eb01dfc9e,2,0,0.099563,0.101146,0.120174,0.116042
3,f454a51a9c08b6bb9021364fe055076d,3,0,0.095299,0.079013,0.185215,0.104300
4,39d85935396f36f884644344b6ec9121,4,0,0.191140,0.302087,0.179507,0.158919
...,...,...,...,...,...,...,...
137061,1a110007fb9569efbcbcdbc48ef7c54f,137061,1,0.729605,0.711259,0.718466,0.770864
137062,cd6a3e9182e07a0b4efc4e210edfd344,137062,1,0.966070,0.937673,0.938973,0.884863
137063,1e71c35e434299170bf6014f168b8a8d,137063,1,0.305158,0.588000,0.333046,0.579210
137064,c64bcf3dda6243a1c9c725300b1ded4f,137064,1,0.315640,0.268119,0.340868,0.314942


## Hard Voting
- using the majority vote of all 4 models - lgbm, xgboost, lr, svm


In [99]:

def hard_vote(sources, id_path):
    """
    sources: dict {model_name: (path, col_name)}
    id_path: parquet file to take 'row_id' from
    """
    def get_prediction(path, col_name, threshold):
        df = pd.read_parquet(path)
        probs = df[col_name]
        preds = np.array([1 if p >= threshold else 0 for p in probs])
        return preds

    names = list(sources)

    final_pred = pd.DataFrame({'row_id': pd.read_parquet(id_path)['row_id']})

    for name, (path, col, threshold) in sources.items():
        final_pred[name] = get_prediction(path, col, threshold)

    # majority voting
    final_pred['hardvoting_pred'] = final_pred[names].mode(axis=1)[0] #default to 0 if number of 0s and 1s is the same
    return final_pred

#hard voting on validation data 
final_pred_val = hard_vote({
    'lgbm':       (val_lgbm,   lgbm_model, 0.5),
    'xgboost':    (val_lgbm,   xgb_model, 0.5),
    'lr':         (val_logreg, lr_model, 0.510238),
    'linear_svm': (val_margin, svm_model, 0.5)},  
    id_path=val_logreg
)

#hard voting on test data 
final_pred_test = hard_vote({
    'lgbm':       (test_lgbm,   lgbm_model, 0.5),
    'xgboost':    (test_lgbm,   xgb_model, 0.5),
    'lr':         (test_logreg, lr_model, 0.510238),
    'linear_svm': (test_margin, svm_model, 0.5)},  
    id_path=test_logreg
)

display(final_pred_test)


,row_id,lgbm,xgboost,lr,linear_svm,hardvoting_pred
0,a46a85087497fefb804b721fc0d3a394,0,0,0,0,0.0
1,3ba780a703bf09a2c05ae47b640d9d73,1,1,0,0,0.0
2,380416409f681ab52f1414e07632f295,0,0,0,0,0.0
3,e6cdfb1978af61d364a9efda8c6aa1de,0,0,0,0,0.0
4,b2c1b05ea686a36460eb1d3d722288ad,0,0,0,0,0.0
...,...,...,...,...,...,...
137060,9d02c1db249e3e7a38d074de9604e305,1,1,1,1,1.0
137061,7fb9334fedcd647999a50531ba5faaae,1,1,1,1,1.0
137062,39f80fcb5f343197d919ee88347baea7,1,1,1,1,1.0
137063,b7ee2ed66f51353fbe7bb54f80112697,1,1,1,1,1.0


## Soft Voting
- using probability of prediction for each model to determine weights of each model in prediction
- weights calculated from validation predictions and then applied on test data 


In [ ]:

def soft_vote(val_sources, test_sources, y_val, id_path=None, threshold=0.5):
    """
    val_sources / test_sources: dict {model_name: (path, col_name)}
        (same model names, in the same order, for both)
    y_val: true validation labels
    id_path: optional test parquet file to take 'row_id' from
    Returns (result_df, scores, weights)
    """
    def get_prob(path, col_name):
        return pd.read_parquet(path)[col_name].values

    names = list(val_sources)
    if list(test_sources) != names:
        raise ValueError("val_sources and test_sources must have the same model names in the same order")

    #calculate weights from validation ROC-AUC
    scores = np.array([
        roc_auc_score(y_val, get_prob(*val_sources[n])) for n in names
    ])
    weights = (scores - 0.5) / (scores - 0.5).sum()

    #apply weights to test probabilities
    P_test = np.column_stack([get_prob(*test_sources[n]) for n in names])
    final_prob = np.average(P_test, axis=1, weights=weights)
    softvoting_preds = (final_prob >= threshold).astype(int)

    #predict for test
    result = pd.DataFrame(P_test, columns=names)
    if id_path is not None:
        result.insert(0, 'row_id', pd.read_parquet(id_path)['row_id'].values)
    result['softvoting_prob'] = final_prob
    result['softvoting_pred'] = softvoting_preds

    return result, pd.Series(scores, index=names), pd.Series(weights, index=names)




result, scores, weights = soft_vote(
    val_sources={
        'lgbm':       (val_lgbm,   lgbm_model),
        'xgboost':    (val_lgbm,   xgb_model),
        'lr':         (val_logreg, lr_model),
        'linear_svm': (val_margin, svm_model),
    },
    test_sources={
        'lgbm':       (test_lgbm,   lgbm_model),
        'xgboost':    (test_lgbm,   xgb_model),   
        'lr':         (test_logreg, lr_model),
        'linear_svm': (test_margin, svm_model),
    },
    y_val=y_val,
    id_path=test_logreg,
)

display(scores)
display(weights)
display(len(result))
display(result)
display(result2)



lgbm          0.804407
xgboost       0.803636
lr            0.800461
linear_svm    0.799682
dtype: float64

lgbm          0.251953
xgboost       0.251316
lr            0.248688
linear_svm    0.248043
dtype: float64

137065

,row_id,lgbm,xgboost,lr,linear_svm,softvoting_prob,softvoting_pred
0,a46a85087497fefb804b721fc0d3a394,0.348674,0.342385,0.382977,0.318301,0.348090,0
1,3ba780a703bf09a2c05ae47b640d9d73,0.502795,0.665451,0.467367,0.412179,0.512386,1
2,380416409f681ab52f1414e07632f295,0.354178,0.393940,0.300995,0.266039,0.329083,0
3,e6cdfb1978af61d364a9efda8c6aa1de,0.026658,0.026994,0.167400,0.211708,0.107644,0
4,b2c1b05ea686a36460eb1d3d722288ad,0.027226,0.053660,0.082605,0.077147,0.060024,0
...,...,...,...,...,...,...,...
137060,9d02c1db249e3e7a38d074de9604e305,0.611933,0.645868,0.717644,0.626183,0.650285,1
137061,7fb9334fedcd647999a50531ba5faaae,0.804859,0.773014,0.853279,0.878472,0.827157,1
137062,39f80fcb5f343197d919ee88347baea7,0.547344,0.526457,0.613528,0.708580,0.598547,1
137063,b7ee2ed66f51353fbe7bb54f80112697,0.690099,0.699844,0.539109,0.659477,0.647403,1


,row_id,lgbm,xgboost,lr,linear_svm,softvoting_prob,softvoting_pred
0,8c5622f4b8edbd4fc1bb29d1a2739d90,0.379181,0.441270,0.561814,0.562795,0.485748,0
1,ef194a1d7209570a5779540806d0be7f,0.425580,0.385260,0.672666,0.674919,0.538741,1
2,0b6b7d6b88a0165663241c9eb01dfc9e,0.101146,0.116042,0.169473,0.142458,0.132129,0
3,f454a51a9c08b6bb9021364fe055076d,0.079013,0.104300,0.170486,0.203085,0.138891,0
4,39d85935396f36f884644344b6ec9121,0.302087,0.158919,0.263975,0.225083,0.237528,0
...,...,...,...,...,...,...,...
137061,1a110007fb9569efbcbcdbc48ef7c54f,0.711259,0.770864,0.631700,0.715480,0.707500,1
137062,cd6a3e9182e07a0b4efc4e210edfd344,0.937673,0.884863,0.804496,0.778818,0.851879,1
137063,1e71c35e434299170bf6014f168b8a8d,0.588000,0.579210,0.654496,0.688755,0.627319,1
137064,c64bcf3dda6243a1c9c725300b1ded4f,0.268119,0.314942,0.420043,0.356135,0.339499,0


## Evaluation


In [87]:
def scores(y, p):
    pred = p 
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"roc_auc": float(roc_auc_score(y, p)),
            "pr_auc": float(average_precision_score(y, p)),
            "pr_auc_definition": "average_precision",
            "accuracy": float(accuracy_score(y, pred)),
            "precision": float(precision_score(y, pred, zero_division=0)),
            "recall": float(recall_score(y, pred, zero_division=0)),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "brier_score": float(brier_score_loss(y, p)),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

In [ ]:

#hard voting evaluation on test 
display(scores(y_test, final_pred_test['hardvoting_pred']))

#soft voting evaluation 
display(scores(y_test, result['softvoting_pred']))






{'roc_auc': 0.7352909650552236,
 'pr_auc': 0.6913859282474482,
 'pr_auc_definition': 'average_precision',
 'accuracy': 0.7336665085908146,
 'precision': 0.7847003824904377,
 'recall': 0.6594285714285715,
 'f1': 0.7166310886862022,
 'brier_score': 0.26633349140918544,
 'tn': 54400,
 'fp': 12665,
 'fn': 23840,
 'tp': 46160}

{'roc_auc': 0.7376864513105622,
 'pr_auc': 0.6878174712931291,
 'pr_auc_definition': 'average_precision',
 'accuracy': 0.737015284718929,
 'precision': 0.7614500877814396,
 'recall': 0.7063428571428572,
 'f1': 0.7328619917885781,
 'brier_score': 0.26298471528107104,
 'tn': 51575,
 'fp': 15490,
 'fn': 20556,
 'tp': 49444}

In [100]:
#check correlation of lgbm and xgboost model
def get_prob(path, col_name):
        return pd.read_parquet(path)[col_name].values

P_val = np.column_stack([get_prob(val_lgbm, lgbm_model), get_prob(val_lgbm , xgb_model)
                         , get_prob(val_logreg, lr_model), get_prob(val_margin, svm_model)])
print(pd.DataFrame(P_val, columns = ['lgbm, xgb, lr, svm']).corr().round(3))

#high corration between lgbm and xgboost models -> errors might be highly correlated
#maybe can use another model



ValueError: Shape of passed values is (137066, 4), indices imply (137066, 1)

In [ ]:
#stacking 
#draw graph
#save models

#oof predictions
#cross validations 
